# 🚀 PaddleOCR v5 Vietnamese - Standalone Kaggle Notebook

**Notebook hoàn chỉnh - KHÔNG CẦN GitHub!**

Tất cả code đã có sẵn trong notebook này. Chỉ cần:
1. Add dataset vào Kaggle
2. Chạy từng cell
3. Done!

---

## ⚙️ Kaggle Settings:
- **Accelerator**: GPU T4 x2
- **Internet**: ON  
- **Add Data**: vietnamese-ocr-250k (dataset của bạn)

## 📊 Dataset Format:
```
vietnamese-ocr-250k/
├── images/
│   ├── img_001.jpg
│   └── ...
└── rec_gt.txt
```

## ⏱️ Timeline:
- Setup: 7 phút
- Test 10k: 30 phút  
- Full training: 16-20 giờ

---

## Cell 1: Create Project Structure

In [ ]:
# Tạo thư mục project
!mkdir -p /kaggle/working/paddleocr-v5-vietnamese
%cd /kaggle/working/paddleocr-v5-vietnamese

!mkdir -p data/train_data data/val_data
!mkdir -p dict output logs inference pretrain_models

print("✅ Project structure created")
!pwd

## Cell 2: Create prepare_data.py

In [ ]:
%%writefile prepare_data.py
#!/usr/bin/env python3
import argparse, os, shutil, random, codecs
from pathlib import Path
from collections import Counter
from tqdm import tqdm

def prepare_kaggle_dataset(input_dir, images_dir='images', label_file='rec_gt.txt', 
                           train_ratio=0.9, max_samples=None, seed=42):
    print("="*70)
    print("🚀 PaddleOCR v5 Vietnamese - Data Preparation")
    print("="*70)
    
    random.seed(seed)
    images_path = os.path.join(input_dir, images_dir)
    label_path = os.path.join(input_dir, label_file)
    
    if not os.path.exists(images_path) or not os.path.exists(label_path):
        print(f"❌ Not found: {images_path} or {label_path}")
        return False
    
    with open(label_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    valid_data = []
    for line in tqdm(lines, desc="Validating"):
        if not line.strip(): continue
        parts = line.strip().split('\t')
        if len(parts) < 2: continue
        img_name, text = parts[0], parts[1]
        img_path = os.path.join(images_path, img_name)
        if os.path.exists(img_path):
            valid_data.append((img_name, text, img_path))
    
    if max_samples and max_samples < len(valid_data):
        random.shuffle(valid_data)
        valid_data = valid_data[:max_samples]
    
    random.shuffle(valid_data)
    split_idx = int(len(valid_data) * train_ratio)
    train_data = valid_data[:split_idx]
    val_data = valid_data[split_idx:]
    
    print(f"\nTrain: {len(train_data):,}, Val: {len(val_data):,}")
    
    os.makedirs('data/train_data', exist_ok=True)
    os.makedirs('data/val_data', exist_ok=True)
    os.makedirs('dict', exist_ok=True)
    
    train_labels = []
    for img_name, text, img_path in tqdm(train_data, desc="Train"):
        shutil.copy2(img_path, f'data/train_data/{img_name}')
        train_labels.append(f"train_data/{img_name}\t{text}\n")
    
    val_labels = []
    for img_name, text, img_path in tqdm(val_data, desc="Val"):
        shutil.copy2(img_path, f'data/val_data/{img_name}')
        val_labels.append(f"val_data/{img_name}\t{text}\n")
    
    with open('data/train_list.txt', 'w', encoding='utf-8') as f:
        f.writelines(train_labels)
    with open('data/val_list.txt', 'w', encoding='utf-8') as f:
        f.writelines(val_labels)
    
    chars = set()
    char_freq = Counter()
    for _, text, _ in valid_data:
        chars.update(text)
        char_freq.update(text)
    
    vietnamese = 'aàáảãạăằắẳẵặâầấẩẫậbcdđeèéẻẽẹêềếểễệfghiìíỉĩịjklmnoòóỏõọôồốổỗộơờớởỡợpqrstuùúủũụưừứửữựvwxyỳýỷỹỵz'
    chars.update(vietnamese + vietnamese.upper() + '0123456789' + ' .,;:!?-_()[]{}/@#$%^&*+=~`\'"<>|\\°§€£¥₫')
    
    sorted_chars = [c for c, _ in char_freq.most_common()] + [c for c in sorted(chars) if c not in char_freq]
    
    with codecs.open('dict/vi_dict.txt', 'w', encoding='utf-8') as f:
        f.write('\n'.join(sorted_chars))
    
    print(f"\n✅ Done! Dictionary: {len(sorted_chars)} chars")
    return True

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--input_dir', required=True)
    parser.add_argument('--images_dir', default='images')
    parser.add_argument('--label_file', default='rec_gt.txt')
    parser.add_argument('--train_ratio', type=float, default=0.9)
    parser.add_argument('--max_samples', type=int, default=None)
    args = parser.parse_args()
    prepare_kaggle_dataset(**vars(args))

## Cell 3: Setup Environment

In [ ]:
# Check GPU
!nvidia-smi

# Install PaddlePaddle
!pip install -q paddlepaddle-gpu==3.0.0b1 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/

# Clone PaddleOCR
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git

# Install dependencies
%cd PaddleOCR
!pip install -q -r requirements.txt
!pip install -q visualdl tqdm
%cd ..

# Download pretrained model
!mkdir -p pretrain_models && cd pretrain_models && \
  wget -q https://paddleocr.bj.bcebos.com/PP-OCRv5/chinese/ch_PP-OCRv5_rec_train.tar && \
  tar -xf ch_PP-OCRv5_rec_train.tar && rm *.tar

print("\n✅ Setup completed!")

## Cell 4: Prepare Data

⚠️ **Sửa dataset name**: Thay `/kaggle/input/vietnamese-ocr-250k` thành tên dataset của bạn!

In [ ]:
# Prepare data
!python prepare_data.py \
    --input_dir /kaggle/input/vietnamese-ocr-250k \
    --images_dir images \
    --label_file rec_gt.txt \
    --train_ratio 0.9

# Verify
!echo "\n📊 Data Summary:"
!wc -l data/*.txt
!head -3 data/train_list.txt

## Cell 5: Create Config

In [ ]:
%%writefile config_kaggle.yml
Global:
  use_gpu: true
  epoch_num: 100
  print_batch_step: 100
  save_model_dir: ./output/vi_ppocr_v5
  save_epoch_step: 5
  eval_batch_step: [0, 2000]
  cal_metric_during_train: true
  pretrained_model: ./pretrain_models/ch_PP-OCRv5_rec_train/best_accuracy
  character_dict_path: ./dict/vi_dict.txt
  max_text_length: &max_text_length 40
  use_space_char: true
  distributed: true

Optimizer:
  name: AdamW
  lr:
    name: Cosine
    learning_rate: 0.0005
    warmup_epoch: 2
  grad_clip:
    name: ClipGradByGlobalNorm
    clip_norm: 5.0

Architecture:
  model_type: rec
  algorithm: SVTR_LCNet
  Backbone:
    name: PPLCNetV3
    scale: 0.95
  Head:
    name: MultiHead
    head_list:
      - CTCHead:
          Neck:
            name: svtr
            dims: 120
            depth: 2
      - NRTRHead:
          nrtr_dim: 384
          max_text_length: *max_text_length
      - SARHead:
          enc_dim: 512
          max_text_length: *max_text_length

Loss:
  name: MultiLoss
  loss_config_list:
    - CTCLoss: {weight: 1.0}
    - NRTRLoss: {weight: 1.0, smoothing: true}
    - SARLoss: {weight: 1.0}

PostProcess:
  name: CTCLabelDecode

Metric:
  name: RecMetric
  main_indicator: acc

Train:
  dataset:
    name: MultiScaleDataSet
    data_dir: ./data/
    label_file_list: [./data/train_list.txt]
    transforms:
      - DecodeImage: {img_mode: BGR}
      - RecAug: {aug_prob: 0.4}
      - RecConAug: {prob: 0.5, image_shape: [48,320,3]}
      - MultiLabelEncode: {gtc_encode: NRTRLabelEncode}
      - KeepKeys: {keep_keys: [image,label_ctc,label_gtc,label_sar,length,valid_ratio]}
  sampler:
    name: MultiScaleSampler
    scales: [[320,32],[320,48],[320,64]]
    first_bs: &bs 96
  loader:
    batch_size_per_card: *bs
    num_workers: 2

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: ./data/
    label_file_list: [./data/val_list.txt]
    transforms:
      - DecodeImage: {img_mode: BGR}
      - MultiLabelEncode: {gtc_encode: NRTRLabelEncode}
      - RecResizeImg: {image_shape: [3,48,320]}
      - KeepKeys: {keep_keys: [image,label_ctc,label_gtc,label_sar,length,valid_ratio]}
  loader:
    batch_size_per_card: 96
    num_workers: 2

## Cell 6: Test 10k (30 phút) ⚡

In [ ]:
# Backup và tạo test subset
!cp data/train_list.txt data/train_full.txt
!cp data/val_list.txt data/val_full.txt
!head -10000 data/train_full.txt > data/train_list.txt
!head -1000 data/val_full.txt > data/val_list.txt

# Create test config
!cp config_kaggle.yml config_test.yml
!sed -i 's/epoch_num: 100/epoch_num: 5/' config_test.yml
!sed -i 's|output/vi_ppocr_v5|output/test_10k|' config_test.yml

# Train test
%cd PaddleOCR
!python -m paddle.distributed.launch --gpus '0,1' \
    tools/train.py -c ../config_test.yml
%cd ..

print("\n✅ Test completed! Check accuracy >70%")

## Cell 7: Full Training (16-20h) 🚀

In [ ]:
# Restore full dataset
!cp data/train_full.txt data/train_list.txt
!cp data/val_full.txt data/val_list.txt

print(f"Train: {!wc -l < data/train_list.txt} samples")

# Full training
%cd PaddleOCR
!python -m paddle.distributed.launch --gpus '0,1' \
    tools/train.py -c ../config_kaggle.yml
%cd ..

## Cell 8: Export Model

In [ ]:
%cd PaddleOCR
!python tools/export_model.py \
    -c ../config_kaggle.yml \
    -o Global.pretrained_model=../output/vi_ppocr_v5/best_accuracy \
       Global.save_inference_dir=../inference
%cd ..

# Package
!tar -czf vi_ppocr_v5_model.tar.gz inference/ dict/
!ls -lh vi_ppocr_v5_model.tar.gz

print("\n✅ Download từ Output tab!")

## 🎉 Done!

Download `vi_ppocr_v5_model.tar.gz` và sử dụng:

```python
from paddleocr import PaddleOCR
ocr = PaddleOCR(rec_model_dir='inference/', rec_char_dict_path='dict/vi_dict.txt')
result = ocr.ocr('test.jpg')
```